In [1]:
!pip install pandas
!pip install matplotlib
!pip install numpy

In [2]:
import pandas as pd
import numpy as np

In [3]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

In [4]:
doc_df = pd.read_csv("raw_data/docindex.csv")
doc_df.head()

,TYPE-COLLATERAL,COLLATERAL,PARTY,DOC-ID,DRDATE,PROCESSING-DATE,CORR-DATE,CORR-ID,SERIAL-ID,DOC-TYPE,Unnamed: 10
0,1,10019,,,20260109,20260109,,,51-0019,BOS,NaN
1,1,10019,,,20260109,20260109,,,51-0019,BOS,NaN
2,1,10034,,,20260109,20260109,,,19045C,BOS,NaN
3,1,10034,,,20260109,20260109,,,19045C,BOS,NaN
4,1,10034,,,20260113,20260113,,,19045C,BOS,NaN


In [5]:
doc_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 15223 entries, 0 to 15222
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   TYPE-COLLATERAL  15223 non-null  int64  
 1   COLLATERAL       15223 non-null  str    
 2   PARTY            15223 non-null  str    
 3   DOC-ID           15223 non-null  str    
 4   DRDATE           15223 non-null  int64  
 5   PROCESSING-DATE  15223 non-null  int64  
 6   CORR-DATE        15223 non-null  str    
 7   CORR-ID          15223 non-null  str    
 8   SERIAL-ID        15223 non-null  str    
 9   DOC-TYPE         15223 non-null  str    
 10  Unnamed: 10      0 non-null      float64
dtypes: float64(1), int64(3), str(7)
memory usage: 1.3 MB


In [6]:
date_cols = ['DRDATE', 'PROCESSING-DATE', 'CORR-DATE']
for col in date_cols:
    if col in doc_df.columns:
        doc_df[col] = pd.to_datetime(doc_df[col], format='%Y%m%d', errors='coerce')

In [7]:
# 1. Clean column names first
doc_df.columns = doc_df.columns.str.strip()

df_obj_colds = doc_df.select_dtypes(['object']).columns

# 3. Apply stripping only to the filtered list
doc_df[df_obj_colds] = doc_df[df_obj_colds].apply(lambda x: x.astype(str).str.strip())

# 4. Drop the ghost column if it exists
if 'Unnamed: 10' in doc_df.columns:
    doc_df = doc_df.drop(columns=['Unnamed: 10'])

/var/folders/r5/7sdqfmvs4p987n2g0n_nng840000gn/T/ipykernel_89296/2378400130.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df_obj_colds = doc_df.select_dtypes(['object']).columns


In [8]:
cols_to_fix = doc_df.columns

# Replace empty/whitespace strings with NaN only in those columns
doc_df[cols_to_fix] = doc_df[cols_to_fix].replace(r'^\s*$', np.nan, regex=True)

In [9]:
doc_df['TYPE-COLLATERAL'].value_counts()

TYPE-COLLATERAL
1    10834
2     2939
9     1113
3      329
5        8
Name: count, dtype: int64

In [10]:
doc_df.columns

Index(['TYPE-COLLATERAL', 'COLLATERAL', 'PARTY', 'DOC-ID', 'DRDATE',
       'PROCESSING-DATE', 'CORR-DATE', 'CORR-ID', 'SERIAL-ID', 'DOC-TYPE'],
      dtype='str')

## Clean `COLLATERAL` column
We introduce 6 more columns, with each column representing each type of collateral. Thus, each row will have 5 NaN values and 1 actual column value for the collateral.

#### From the FAA dataset description:

Contents are based on type collateral
code, as follows:
* 1 - Aircraft N-Number
* 2 - Engine Make, Model and Serial Number
* 3 - Propeller Make, Model and Serial Number
* 4 - Spare Parts Location
* 5 - “DOC”, followed by recordation number
* 9 - “UNIDENTIFIED” or a partial description of the document



In [11]:
mapping = {
    1: 'Aircraft',
    2: 'Engine',
    3: 'Propeller',
    4: 'Spare_Parts',
    5: 'Document',
    9: 'Unidentified'
}

for code, name in mapping.items():
    column_name = f"Collateral_{name}"

    # Get the base data for this specific type
    values = doc_df['COLLATERAL'].where(doc_df['TYPE-COLLATERAL'] == code)

    # If it's the Aircraft type (code 1), prepend "N" to the non-null values
    if code == 1:
        values = values.apply(lambda x: f"N{x}" if pd.notnull(x) else x)

    doc_df[column_name] = values

doc_df.drop(columns=['COLLATERAL'], inplace=True)

In [12]:
doc_df

,TYPE-COLLATERAL,PARTY,DOC-ID,DRDATE,PROCESSING-DATE,CORR-DATE,CORR-ID,SERIAL-ID,DOC-TYPE,Collateral_Aircraft,Collateral_Engine,Collateral_Propeller,Collateral_Spare_Parts,Collateral_Document,Collateral_Unidentified
0,1,NaN,NaN,2026-01-09,2026-01-09,NaT,NaN,51-0019,BOS,N10019,NaN,NaN,NaN,NaN,NaN
1,1,NaN,NaN,2026-01-09,2026-01-09,NaT,NaN,51-0019,BOS,N10019,NaN,NaN,NaN,NaN,NaN
2,1,NaN,NaN,2026-01-09,2026-01-09,NaT,NaN,19045C,BOS,N10034,NaN,NaN,NaN,NaN,NaN
3,1,NaN,NaN,2026-01-09,2026-01-09,NaT,NaN,19045C,BOS,N10034,NaN,NaN,NaN,NaN,NaN
4,1,NaN,NaN,2026-01-13,2026-01-13,NaT,NaN,19045C,BOS,N10034,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15218,9,NaN,NaN,2026-01-08,2026-01-08,2026-01-08,C,18736P21496,BOS,NaN,NaN,NaN,NaN,NaN,ZIPLINE INTERNATIONAL INC P2 ZIP 1873
15219,9,NaN,NaN,2026-01-08,2026-01-08,2026-01-08,C,18736P21497,BOS,NaN,NaN,NaN,NaN,NaN,ZIPLINE INTERNATIONAL INC P2 ZIP 1873
15220,9,NaN,NaN,2026-01-08,2026-01-08,2026-01-08,C,18736P21498,BOS,NaN,NaN,NaN,NaN,NaN,ZIPLINE INTERNATIONAL INC P2 ZIP 1873
15221,9,NaN,NaN,2026-01-08,2026-01-08,2026-01-08,C,18736P21499,BOS,NaN,NaN,NaN,NaN,NaN,ZIPLINE INTERNATIONAL INC P2 ZIP 1873


## Clean `TYPE-COLLATERAL` column
From the FAA dataset description:
* 1 - Aircraft
* 2 - Engine
* 3 - Propeller
* 4 - Spare Parts
* 5 - Document
* 9 - Unidentified

In [13]:
doc_df['TYPE-COLLATERAL'].value_counts(dropna=False)

TYPE-COLLATERAL
1    10834
2     2939
9     1113
3      329
5        8
Name: count, dtype: int64

In [14]:
collateral_mapping = {
    1: 'Aircraft',
    2: 'Engine',
    3: 'Propeller',
    4: 'Spare Parts',
    5: 'Document',
    9: 'Unidentified'
}

doc_df['TYPE-COLLATERAL'] = doc_df['TYPE-COLLATERAL'].map(collateral_mapping)

In [15]:
doc_df['TYPE-COLLATERAL'].value_counts(dropna=False)

TYPE-COLLATERAL
Aircraft        10834
Engine           2939
Unidentified     1113
Propeller         329
Document            8
Name: count, dtype: int64

## Cleaning `CORR-ID` column
From the FAA dataset description, we know:

Denotes if record has been added or corrected.

A - Record was added

C - Record was changed

But, we don't know what `D` stands for in this column. There are 43 such values. We will leave them as is.

In [16]:
doc_df['CORR-ID'].value_counts(dropna=False)

CORR-ID
NaN    13723
C       1280
A        177
D         43
Name: count, dtype: int64

In [17]:
corr_mapping = {
    'A': 'Record was added',
    'C': 'Record was changed'
}

doc_df['CORR-ID'] = doc_df['CORR-ID'].map(corr_mapping).fillna(doc_df['CORR-ID'])

In [18]:
doc_df['CORR-ID'].value_counts(dropna=False)

CORR-ID
NaN                   13723
Record was changed     1280
Record was added        177
D                        43
Name: count, dtype: int64

In [19]:
doc_df

,TYPE-COLLATERAL,PARTY,DOC-ID,DRDATE,PROCESSING-DATE,CORR-DATE,CORR-ID,SERIAL-ID,DOC-TYPE,Collateral_Aircraft,Collateral_Engine,Collateral_Propeller,Collateral_Spare_Parts,Collateral_Document,Collateral_Unidentified
0,Aircraft,NaN,NaN,2026-01-09,2026-01-09,NaT,NaN,51-0019,BOS,N10019,NaN,NaN,NaN,NaN,NaN
1,Aircraft,NaN,NaN,2026-01-09,2026-01-09,NaT,NaN,51-0019,BOS,N10019,NaN,NaN,NaN,NaN,NaN
2,Aircraft,NaN,NaN,2026-01-09,2026-01-09,NaT,NaN,19045C,BOS,N10034,NaN,NaN,NaN,NaN,NaN
3,Aircraft,NaN,NaN,2026-01-09,2026-01-09,NaT,NaN,19045C,BOS,N10034,NaN,NaN,NaN,NaN,NaN
4,Aircraft,NaN,NaN,2026-01-13,2026-01-13,NaT,NaN,19045C,BOS,N10034,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15218,Unidentified,NaN,NaN,2026-01-08,2026-01-08,2026-01-08,Record was changed,18736P21496,BOS,NaN,NaN,NaN,NaN,NaN,ZIPLINE INTERNATIONAL INC P2 ZIP 1873
15219,Unidentified,NaN,NaN,2026-01-08,2026-01-08,2026-01-08,Record was changed,18736P21497,BOS,NaN,NaN,NaN,NaN,NaN,ZIPLINE INTERNATIONAL INC P2 ZIP 1873
15220,Unidentified,NaN,NaN,2026-01-08,2026-01-08,2026-01-08,Record was changed,18736P21498,BOS,NaN,NaN,NaN,NaN,NaN,ZIPLINE INTERNATIONAL INC P2 ZIP 1873
15221,Unidentified,NaN,NaN,2026-01-08,2026-01-08,2026-01-08,Record was changed,18736P21499,BOS,NaN,NaN,NaN,NaN,NaN,ZIPLINE INTERNATIONAL INC P2 ZIP 1873


## Cleaning `DOC-TYPE` column

From the FAA dataset description, we know:

* BOS – Evidence of Ownership
* S/A – Security Conveyance/Lien
* REL – Lien Release
* REP – Repossession
* CSC – Evidence of Ownership w/Lien
* DIS – Disclaimer
* CRT – Court Documents

In [20]:
doc_df['DOC-TYPE'].value_counts(dropna=False)

DOC-TYPE
BOS    8248
S/A    3785
REL    2891
DIS     184
CRT      50
CSC      42
REP      23
Name: count, dtype: int64

In [21]:
# Define the Document Type mapping
doc_type_mapping = {
    'BOS': 'Evidence of Ownership',
    'S/A': 'Security Conveyance/Lien',
    'REL': 'Lien Release',
    'REP': 'Repossession',
    'CSC': 'Evidence of Ownership w/Lien',
    'DIS': 'Disclaimer',
    'CRT': 'Court Documents'
}

doc_df['DOC-TYPE'] = doc_df['DOC-TYPE'].map(doc_type_mapping)

In [22]:
doc_df['DOC-TYPE'].value_counts(dropna=False)

DOC-TYPE
Evidence of Ownership           8248
Security Conveyance/Lien        3785
Lien Release                    2891
Disclaimer                       184
Court Documents                   50
Evidence of Ownership w/Lien      42
Repossession                      23
Name: count, dtype: int64

## Final Dataset Cleaning

In [23]:
collateral_cols = [col for col in doc_df.columns if col.startswith('Collateral_')]

remaining_cols = [col for col in doc_df.columns if col not in collateral_cols and col != 'TYPE-COLLATERAL']

# 3. Combine into the new desired order
new_column_order = ['TYPE-COLLATERAL'] + collateral_cols + remaining_cols

# 4. Re-index the DataFrame
doc_df = doc_df[new_column_order]

In [24]:
doc_df

,TYPE-COLLATERAL,Collateral_Aircraft,Collateral_Engine,Collateral_Propeller,Collateral_Spare_Parts,Collateral_Document,Collateral_Unidentified,PARTY,DOC-ID,DRDATE,PROCESSING-DATE,CORR-DATE,CORR-ID,SERIAL-ID,DOC-TYPE
0,Aircraft,N10019,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-01-09,2026-01-09,NaT,NaN,51-0019,Evidence of Ownership
1,Aircraft,N10019,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-01-09,2026-01-09,NaT,NaN,51-0019,Evidence of Ownership
2,Aircraft,N10034,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-01-09,2026-01-09,NaT,NaN,19045C,Evidence of Ownership
3,Aircraft,N10034,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-01-09,2026-01-09,NaT,NaN,19045C,Evidence of Ownership
4,Aircraft,N10034,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-01-13,2026-01-13,NaT,NaN,19045C,Evidence of Ownership
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15218,Unidentified,NaN,NaN,NaN,NaN,NaN,ZIPLINE INTERNATIONAL INC P2 ZIP 1873,NaN,NaN,2026-01-08,2026-01-08,2026-01-08,Record was changed,18736P21496,Evidence of Ownership
15219,Unidentified,NaN,NaN,NaN,NaN,NaN,ZIPLINE INTERNATIONAL INC P2 ZIP 1873,NaN,NaN,2026-01-08,2026-01-08,2026-01-08,Record was changed,18736P21497,Evidence of Ownership
15220,Unidentified,NaN,NaN,NaN,NaN,NaN,ZIPLINE INTERNATIONAL INC P2 ZIP 1873,NaN,NaN,2026-01-08,2026-01-08,2026-01-08,Record was changed,18736P21498,Evidence of Ownership
15221,Unidentified,NaN,NaN,NaN,NaN,NaN,ZIPLINE INTERNATIONAL INC P2 ZIP 1873,NaN,NaN,2026-01-08,2026-01-08,2026-01-08,Record was changed,18736P21499,Evidence of Ownership


In [25]:
doc_df.columns = (doc_df.columns
                     .str.replace(' ', '_', regex=False)
                     .str.replace('-', '_', regex=False)
                     .str.replace('(', '', regex=False)
                     .str.replace(')', '', regex=False)
                     .str.lower())

In [26]:
doc_df.to_csv('./clean_data/DOCINDEX.csv', index=False, na_rep='')